> **Archived 2025 notebook (outputs removed).** Kept for provenance only; its logs are in `results/legacy_2025/`. Use `training/notebooks/01_train_kaggle.ipynb` for the controlled study.


In [ ]:
# clone YOLOv5 repository

!git clone https://github.com/aash1999/yolov5-cbam/  # clone repo


In [ ]:
%cd yolov5-cbam

In [ ]:
# install dependencies as necessary

!pip install -qr requirements.txt  # install dependencies (ignore errors)

import torch



from IPython.display import Image, clear_output  # to display images

from utils.downloads import attempt_download  # to download models/datasets



# clear_output()

print('Setup complete. Using torch %s %s' % (torch.__version__, torch.cuda.get_device_properties(0) if torch.cuda.is_available() else 'CPU'))

In [ ]:
import os
!pip install roboflow


from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])

project = rf.workspace("dronecv").project("mineguard-thermal")

version = project.version(15)

dataset = version.download("yolov5")

In [ ]:
%pwd

In [ ]:
dataset.location

In [ ]:

%cat {dataset.location}/data.yaml

# Define Model Configuration and Architecture



We will write a yaml script that defines the parameters for our model like the number of classes, anchors, and each layer.



You do not need to edit these cells, but you may.

In [ ]:

import yaml

with open(dataset.location + "/data.yaml", 'r') as stream:

    num_classes = str(yaml.safe_load(stream)['nc'])

In [ ]:
num_classes

In [ ]:
%cat /kaggle/working/yolov5-cbam/models/custom_yolov5x.yaml

# Train Custom YOLOv5 Detector



### Next, we'll fire off training!





Here, we are able to pass a number of arguments:

- **img:** define input image size

- **batch:** determine batch size

- **epochs:** define the number of training epochs. (Note: often, 3000+ are common here!)

- **data:** set the path to our yaml file

- **cfg:** specify our model configuration

- **weights:** specify a custom path to weights. (Note: you can download weights from the Ultralytics Google Drive [folder](https://drive.google.com/open?id=1Drs_Aiu7xx6S-ix95f9kNsA6ueKRpN2J))

- **name:** result names

- **nosave:** only save the final checkpoint

- **cache:** cache images for faster training

In [ ]:
## train yolov5s on custom data for 200 epochs

# time its performance

# %%time


%cd /kaggle/working/yolov5-cbam/

!python train.py --img 416 --batch 16 --epochs 200 --data {dataset.location}/data.yaml --cfg models/yolo5m-cbam-involution.yaml --weights 'yolov5x.pt' --name yolov5s_results  --cache

# Evaluate Custom YOLOv5 Detector Performance

You can view the training graphs associated with a training job in the `/content/yolov5/runs/train/yolov5s_results/results.png` folder.



Training losses and performance metrics are also saved to Tensorboard and also to a logfile defined above with the **--name** flag when we train. In our case, we named this `yolov5s_results`.



Note from Glenn: Partially completed `results.txt` files can be plotted with `from utils.utils import plot_results; plot_results()`.

In [ ]:
from utils.plots import plot_results  # plot results.txt as results.png

Image(filename='/kaggle/working/yolov5-cbam/runs/train/yolov5s_results/results.png', width=1000)  # view results.png

### Curious? Visualize Our Training Data with Labels



After training starts, view `train*.jpg` images to see training images, labels and augmentation effects.



Note a mosaic dataloader is used for training (shown below), a new dataloading concept developed by Glenn Jocher and first featured in [YOLOv4](https://arxiv.org/abs/2004.10934).

In [ ]:
Image(filename='/kaggle/working/yolov5-cbam/runs/train/yolov5s_results/val_batch0_labels.jpg', width=900)

In [ ]:
# print out an augmented training example

print("GROUND TRUTH AUGMENTED TRAINING DATA:")

Image(filename='/kaggle/working/yolov5-cbam/runs/train/yolov5s_results/train_batch0.jpg', width=900)

# Run Inference With Trained Weights



Next, we can run inference with a pretrained checkpoint on all images in the `test/images` folder to understand how our model performs on our test set.

In [ ]:
# trained weights are saved by default in our weights folder

%ls runs/

In [ ]:
%ls runs/train/yolov5s_results/weights

In the snippet below, replace `Cash-Counter-10` with the name of the folder in which your dataset is stored.

In [ ]:
%cd /kaggle/working/yolov5-cbam

!python detect.py --weights /kaggle/working/yolov5-cbam/runs/train/yolov5s_results/weights/best.pt --img 416 --conf 0.4 --source /kaggle/working/yolov5-cbam/MineGuard-Thermal-15/test/images

In [ ]:
import glob

from IPython.display import Image, display



for imageName in glob.glob('/kaggle/working/yolov5-cbam/runs/detect/exp2/*.jpg')[:10]: #assuming JPG

    display(Image(filename=imageName))